In [ ]:
import pandas as pd
import numpy as np


mobility = pd.read_csv("data/va_mobility.csv")
mobility['date'] = pd.to_datetime(mobility['date'])

na_ratio = mobility.isna().mean()
print(na_ratio)

features = [
    'retail_and_recreation_percent_change_from_baseline', 
    'grocery_and_pharmacy_percent_change_from_baseline', 
    'parks_percent_change_from_baseline', 
    'transit_stations_percent_change_from_baseline', 
    'workplaces_percent_change_from_baseline', 
    'residential_percent_change_from_baseline'
]

rural_counties = ["Accomack County","Alleghany County","Bath County",
                  "Bland County","Brunswick County","Buchanan County",
                  "Carroll County","Charlotte County","Craig County",
                  "Dickenson County","Essex County","Grayson County",
                  "Greensville County","Halifax County","Henry County",
                  "Highland County","Lee County","Louisa County",
                  "Lunenburg County","Madison County","Mecklenburg County",
                  "Middlesex County","Montgomery County","Nelson County",
                  "Northampton County","Northumberland County","Patrick County",
                  "Pittsylvania County","Prince Edward County","Pulaski County",
                  "Richmond County","Rockbridge County","Rockingham County",
                  "Russell County","Smyth County","Southampton County",
                  "Tazewell County","Wise County","Wythe County","Shenandoah County"]

mobility['metro_label'] = np.where(mobility['sub_region_2'].isin(rural_counties), 0, 1)

In [ ]:
features_keep = [f for f in features if mobility[f].notna().mean() > 0.3]

print("Keeping features:", features_keep)

In [ ]:
county_max_missing = (
    mobility
    .groupby('sub_region_2')[features_keep]
    .apply(lambda df: df.isna().mean().max())
)
counties_keep = county_max_missing[county_max_missing < 0.5].index

mobility_counties_filtered = mobility[mobility['sub_region_2'].isin(counties_keep)]


In [ ]:
mobility_clean = (
    mobility_counties_filtered
    .sort_values(['sub_region_2', 'date'])
    .groupby('sub_region_2', group_keys=False)
    .apply(lambda df: df[features_keep].interpolate(limit_direction='both')
                          .ffill()
                          .bfill()
                          .assign(metro_label=df['metro_label'], date=df['date'], sub_region_2=df['sub_region_2']))
)

In [ ]:
print(len(mobility_clean['sub_region_2'].unique()))
print(mobility_clean[:].isna().sum())

print(mobility_clean['metro_label'].sum()/len(mobility_clean))

mobility_clean.to_csv("data/mobility_clean.csv", index=False)

In [ ]:
mobility_1 = mobility_clean.copy()
mobility_1["year_month"] = mobility_1["date"].dt.to_period('M')

monthly = mobility_1.groupby("year_month")[features_keep].mean()
monthly.to_csv("data/monthly.csv")